<a href="https://colab.research.google.com/github/tadapin/tidb-colab-tutorials/blob/main/tutorials/07_memory.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# AI エージェントの永続メモリ (ユーザー × 発言ログ)

このノートブックは **pytidb シリーズの第 7 回** です。ベクトル検索を使って AI エージェントに「過去の文脈を思い出す」能力を付与します。

## 学習目標
- `user_id` でマルチユーザー対応した `memories` テーブルを作る
- 新しい発言から事実を抽出 (`google.colab.ai.generate_text`) し、ベクトル化して保存
- 質問に応じて過去メモリをベクトル検索で呼び戻し、LLM プロンプトに差し込む
- RAG (外部ドキュメント) と memory (対話履歴) の違いを理解する

前提: `03_vector_search.ipynb`、`06_rag.ipynb` を実行済み。

本ノートブックはLLMのモデルとして、Google Colabで利用できるGeminiを利用しています。 (`google.colab.ai` を利用)。


## 1. 依存ライブラリのインストールとTiDB Cloudクラスタの作成


In [ ]:
!pip install -q pytidb


In [ ]:
from pathlib import Path
import requests

# TiDB Cloud Zero のAPIエンドポイント (サインアップ不要、30日有効)
ZERO_API = "https://zero.tidbapi.com/v1beta1/instances"


def request_zero_instance(tag: str = "pytidb-tutorial") -> dict:
    """TiDB Cloud Zero のインスタンスを作成して接続情報を返す"""
    resp = requests.post(ZERO_API, json={"tag": tag}, timeout=30)
    resp.raise_for_status()
    return resp.json()["instance"]


instance = request_zero_instance(tag="pytidb-memory")
conn = instance["connection"]
claim_url = instance.get("claimInfo", {}).get("claimUrl", "(取得失敗)")
expires_at = instance.get("expiresAt", "(取得失敗)")

print("=== プロビジョニング完了 ===")
print(f"Host     : {conn['host']}")
print(f"User     : {conn['username']}")
print(f"Expires  : {expires_at}")
print()
print("インスタンスは 30 日後に自動削除されます。")
print("保持したい場合は claim URL を開いてください:")
print(claim_url)


## 2. メモリテーブル

1行 = 1つの「覚えておきたい事実」となるよう、Memoryテーブルを作成します。
`user_id` でマルチユーザーに対応します。`content` は TEXT、自動埋め込みで `content_vec` が生成されます。


In [ ]:
import datetime
from pytidb import TiDBClient
from pytidb.datatype import TEXT
from pytidb.embeddings import EmbeddingFunction
from pytidb.schema import Field, TableModel

db = TiDBClient.connect(
    host=conn["host"],
    port=4000,
    username=conn["username"],
    password=conn["password"],
    database="memory_demo",
    ensure_db=True,
)

_embed = EmbeddingFunction(model_name="tidbcloud_free/amazon/titan-embed-text-v2")


class Memory(TableModel):
    __tablename__ = "memories"
    __table_args__ = {"extend_existing": True}

    id: int = Field(default=None, primary_key=True)
    user_id: str = Field(max_length=32)
    content: str = Field(sa_type=TEXT)
    content_vec: list[float] = _embed.VectorField(source_field="content")
    created_at: datetime.datetime = Field(default_factory=datetime.datetime.utcnow)


table = db.create_table(schema=Memory, if_exists="overwrite")
print("テーブル準備OK:", table.table_name)


## 3. 事実抽出 + 保存のヘルパー関数

会話スニペットから「覚えておくべき事実」を抽出し、ユーザーごとに保存します。
抽出には `google.colab.ai.generate_text()` を使います。


In [ ]:
from google.colab import ai

EXTRACT_PROMPT = """以下のユーザー発言から、今後の会話で思い出すと有益な「事実」を 1 行 1 件、箇条書きにしてください。
事実以外の説明や記号は入れないでください。該当がなければ空行のみを返してください。

ユーザー発言:
{utterance}
"""


def extract_and_store(user_id: str, utterance: str) -> list[str]:
    """ユーザーの発言から事実を抽出してメモリに保存する"""
    raw = ai.generate_text(EXTRACT_PROMPT.format(utterance=utterance))
    facts = [line.lstrip("-・ ").strip() for line in raw.splitlines() if line.strip()]
    for f in facts:
        table.insert(Memory(user_id=user_id, content=f))
    return facts


def recall(user_id: str, query: str, limit: int = 3) -> list[str]:
    """ユーザーごとに関連メモリをベクトル検索で思い出す"""
    results = (
        table.search(query)
        .filter({"user_id": user_id})
        .limit(limit)
        .to_pydantic()
    )
    return [r.hit.content for r in results]


## 4. 2 人分の会話履歴を投入する

Alice と Bob の過去発言を入れて、それぞれの事実を抽出・保存します。


In [ ]:
alice_utterances = [
    "私は Python を 5 年くらい書いています。仕事ではデータ分析が中心です。",
    "コーヒーはブラック派で、朝はエスプレッソを 1 杯飲みます。",
    "最近は Tokyo で開催される PyCon JP に毎年参加しています。",
]
bob_utterances = [
    "I am working on a Go backend for an e-commerce startup in Berlin.",
    "I prefer tea over coffee, especially Japanese green tea.",
    "I attended KubeCon in Europe last year and gave a short talk.",
]
for u in alice_utterances:
    facts = extract_and_store("alice", u)
    print(f"[alice] {len(facts)} fact(s) stored")
for u in bob_utterances:
    facts = extract_and_store("bob", u)
    print(f"[bob]   {len(facts)} fact(s) stored")
print(f"合計メモリ: {table.rows()} 件")


## 5. 呼び戻し + LLM 応答

ユーザーごとに、質問に関連するメモリを取り出してプロンプトに埋め込みます。
同じ質問でも、ユーザーによって使えるメモリが異なるので回答も変わります。


In [ ]:
ANSWER_PROMPT = """あなたはユーザーの好みを知っているアシスタントです。
以下の「メモリ」だけを根拠に、ユーザーの質問へ日本語で丁寧に答えてください。

--- メモリ ---
{memories}
--- ここまで ---

質問: {question}
"""


def answer(user_id: str, question: str) -> str:
    memories = recall(user_id, question, limit=4)
    mem_text = "\n".join(f"- {m}" for m in memories) or "(該当メモリなし)"
    prompt = ANSWER_PROMPT.format(memories=mem_text, question=question)
    return ai.generate_text(prompt).strip()


question = "このユーザーにおすすめの飲み物は何ですか?"
for uid in ("alice", "bob"):
    print(f"=== {uid} ===")
    print(answer(uid, question))
    print()


## チャレンジ
- 新しい発言 (`extract_and_store`) を追加したあとに再度質問してみる
- `recall(..., limit=1)` で 1 件だけ呼び戻して、回答の具体性がどう変わるか見る
- `created_at` で期間フィルタ (`filter={"created_at": {"$gte": ...}}`) をかけ、最近の記憶だけを使う
